In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Path to your folders
icu_path = '/content/drive/MyDrive/MIMIC_ICU/'
hosp_path = '/content/drive/MyDrive/MIMIC_HOSP/'

# Load the "Mega" hospital sheet
mega = pd.read_csv(hosp_path + 'MIMIC_ULTIMATE_MAX_RECORDS.csv')

# Load the "Master" ICU sheet
# (Make sure you saved master_icu to CSV in your previous workbook first!)
master_icu = pd.read_excel(icu_path + 'ICU_Master_Data.xlsx')

In [ ]:
import pandas as pd

In [ ]:
import pandas as pd
from google.colab import drive
import os

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Define Paths based on your screenshots
hosp_path = '/content/drive/MyDrive/MIMIC_HOSP/'
icu_path = '/content/drive/MyDrive/MIMIC_ICU/'

# 3. Load your pre-processed Hospital Master Sheet
print("Loading Hospital Mega Sheet...")
mega = pd.read_csv(hosp_path + 'MIMIC_ULTIMATE_MAX_RECORDS.csv')

# 4. Define Cardiac Keywords (Heart, Coronary, Failure, Arrest, etc.)
cardiac_keywords = 'heart|cardiac|coronary|infarction|failure|arrest|tachycardia|arrhythmia'

# Create a flag: True if the diagnosis title contains any cardiac keywords
# 'long_title' comes from your d_icd_diagnoses merge in the previous workbook
mega['is_cardiac'] = mega['long_title'].str.contains(cardiac_keywords, case=False, na=False)

# Filter for only Cardiac Records
cardiac_hosp_df = mega[mega['is_cardiac'] == True].copy()
print(f"Found {len(cardiac_hosp_df)} cardiac-related records in Hospital data.")

# 5. Load the ICU Stays to link them
print("Loading ICU stays...")
icu_stays = pd.read_csv(icu_path + 'icustays.csv.gz', compression='gzip')

# 6. Merge to get Cardiac ICU stays
# This identifies which ICU stays belong to cardiac patients
cardiac_icu_stays = pd.merge(cardiac_hosp_df, icu_stays, on=['subject_id', 'hadm_id'], how='inner')

# 7. Extract Cardiac Vitals from Chartevents
# We load d_items to find exactly which ItemIDs are 'Heart Rate' etc.
d_items = pd.read_csv(icu_path + 'd_items.csv.gz')
heart_rate_items = d_items[d_items['label'].str.contains('heart rate', case=False, na=False)]['itemid'].tolist()

print("Loading and filtering Chartevents (this may take a moment)...")
# We only load chartevents for the cardiac stay_ids to save RAM
chunks = pd.read_csv(icu_path + 'chartevents.csv.gz', compression='gzip', chunksize=100000)
cardiac_vitals_list = []

for chunk in chunks:
    # Filter chunk for cardiac stay_ids AND heart rate item_ids
    filtered_chunk = chunk[(chunk['stay_id'].isin(cardiac_icu_stays['stay_id'])) &
                           (chunk['itemid'].isin(heart_rate_items))]
    cardiac_vitals_list.append(filtered_chunk)

cardiac_vitals_df = pd.concat(cardiac_vitals_list)

# 8. Final Export
cardiac_icu_stays.to_csv(hosp_path + 'CARDIAC_PATIENTS_SUMMARY.csv', index=False)
print(f"SUCCESS: Extracted {len(cardiac_icu_stays)} cardiac ICU stays.")
print(f"Saved to: {hosp_path}CARDIAC_PATIENTS_SUMMARY.csv")

# Show top results
cardiac_icu_stays[['subject_id', 'hadm_id', 'long_title', 'total_los_days']].head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading Hospital Mega Sheet...
Found 222 cardiac-related records in Hospital data.
Loading ICU stays...
Loading and filtering Chartevents (this may take a moment)...
SUCCESS: Extracted 119 cardiac ICU stays.
Saved to: /content/drive/MyDrive/MIMIC_HOSP/CARDIAC_PATIENTS_SUMMARY.csv


,subject_id,hadm_id,long_title,total_los_days
0,10002428,20321825.0,Acute respiratory failure,2.834028
1,10002428,20321825.0,Acute respiratory failure,2.834028
2,10002428,20321825.0,Acute respiratory failure,2.834028
3,10002428,20321825.0,Acute respiratory failure,2.834028
4,10002428,20321825.0,Acute respiratory failure,2.834028


In [ ]:
import pandas as pd

# 1. Load the cardiac summary created in the previous step
file_path = '/content/drive/MyDrive/MIMIC_HOSP/CARDIAC_PATIENTS_SUMMARY.csv'
df = pd.read_csv(file_path)

# 2. FEATURE REMOVAL
# Dropping the specific blood pressure variants and demographic features requested
features_to_remove = [
    'Blood Pressure', 'Blood Pressure Lying', 'Blood Pressure Sitting',
    'Blood Pressure Standing (1 min)', 'Blood Pressure Standing (3 mins)',
    'insurance', 'language', 'marital_status', 'race'
]

# Use errors='ignore' so the code doesn't crash if one column is already missing
df = df.drop(columns=features_to_remove, errors='ignore')

# 3. CONSOLIDATE CARE UNITS
# Combining First and Last care unit into one "Care_Path" feature
df['unit_path'] = df['first_careunit'] + " -> " + df['last_careunit']
# Remove the old individual columns
df = df.drop(columns=['first_careunit', 'last_careunit'])

# 4. CALCULATE ICU DURATION (from intime_y and outtime_y)
df['intime_y'] = pd.to_datetime(df['intime_y'])
df['outtime_y'] = pd.to_datetime(df['outtime_y'])

# Create a new feature for total ICU hours
df['total_icu_hours'] = (df['outtime_y'] - df['intime_y']).dt.total_seconds() / 3600
# Rounding to 2 decimal places for cleanliness
df['total_icu_hours'] = df['total_icu_hours'].round(2)

# Remove the raw timestamps to keep the sheet clean
df = df.drop(columns=['intime_y', 'outtime_y'])

# 5. CREATE THE NEW EXCEL SHEET
output_path = '/content/drive/MyDrive/MIMIC_HOSP/CARDIAC_REFINED_ANALYTICS.xlsx'

# Using ExcelWriter to ensure a clean export
df.to_excel(output_path, index=False)

print(f"Refinement Complete!")
print(f"New file saved as: {output_path}")
print(f"Remaining Features: {list(df.columns)}")

Refinement Complete!
New file saved as: /content/drive/MyDrive/MIMIC_HOSP/CARDIAC_REFINED_ANALYTICS.xlsx
Remaining Features: ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'edregtime', 'edouttime', 'hospital_expire_flag', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod', 'transfer_id', 'eventtype', 'careunit', 'intime_x', 'outtime_x', 'long_title', 'BMI (kg/m2)', 'Height (Inches)', 'Weight (Lbs)', 'total_med_events', 'max_drg_severity', 'total_los_days', 'ed_boarding_hours', 'is_cardiac', 'stay_id', 'los', 'unit_path', 'total_icu_hours']


In [ ]:
import pandas as pd
import numpy as np

# 1. Load the previous sheet
# Make sure the path matches where you saved the last one
path = '/content/drive/MyDrive/MIMIC_HOSP/'
df = pd.read_excel(path + 'CARDIAC_REFINED_ANALYTICS.xlsx')

# 2. CREATE THE "SURVIVE" COLUMN (The Label)
# hospital_expire_flag: 1 means died in hospital, 0 means survived
# We will create a 'death_event' column where 1 = Died, 0 = Survived
df['death_event'] = df['hospital_expire_flag'].fillna(0).astype(int)

# 3. FEATURE ENGINEERING (The "Handsome" Features)
# A. Care Intensity: Meds per hour in ICU
# (Avoid division by zero by adding a tiny number)
df['med_intensity_per_hour'] = df['total_med_events'] / (df['total_icu_hours'] + 0.1)

# B. ED Stress Ratio: % of total hospital time spent waiting in ED
# Total LOS is in days, so we convert ED hours to days
df['ed_wait_ratio'] = (df['ed_boarding_hours'] / 24) / (df['total_los_days'] + 0.1)

# C. Admission-Discharge Window (Concatenated Time Range)
# Combining dates into a string to see the "Stay Window"
df['admit_time'] = pd.to_datetime(df['admittime']).dt.strftime('%Y-%m-%d')
df['disch_time'] = pd.to_datetime(df['dischtime']).dt.strftime('%Y-%m-%d')
df['stay_window'] = df['admit_time'] + " to " + df['disch_time']

# 4. REMOVE THE "NOISE" (Administrative & Redundant Columns)
cols_to_remove = [
    'admit_provider_id', 'admission_location', 'discharge_location',
    'anchor_year', 'anchor_year_group', 'transfer_id', 'eventtype',
    'careunit', 'hospital_expire_flag', 'edregtime', 'edouttime',
    'admittime', 'dischtime', 'deathtime', 'dod'
]

df_final = df.drop(columns=cols_to_remove, errors='ignore')

# 5. FINAL CLEANUP
# Rounding the new handsome features for readability
df_final['med_intensity_per_hour'] = df_final['med_intensity_per_hour'].round(3)
df_final['ed_wait_ratio'] = df_final['ed_wait_ratio'].round(3)

# 6. SAVE AS PART 2
output_file = path + 'CARDIAC_REFINED_ANALYTICS_PART2.xlsx'
df_final.to_excel(output_file, index=False)

print(f"Success! 'Cardiac Refine Analytics Part 2' created.")
print(f"Final Row Count: {len(df_final)}")
print(f"Key Features Created: ['death_event', 'med_intensity_per_hour', 'ed_wait_ratio', 'stay_window']")

Success! 'Cardiac Refine Analytics Part 2' created.
Final Row Count: 119
Key Features Created: ['death_event', 'med_intensity_per_hour', 'ed_wait_ratio', 'stay_window']
